# 🔭 Fase 5 — Resumo Automático e Busca Inteligente
## Sumarização + Busca Semântica com NorBERTo e LLMs

---
**Roteiro-Desafio-NLP · Fatec DSM 6º Semestre · 2026**  
**Grupo:** ___________________________  **Data:** ___/___/______

---

### 🎯 Objetivo desta fase
Construir um sistema de **busca semântica** usando os embeddings do NorBERTo como retriever, combinado com uma API de LLM para gerar respostas sumarizadas. Esta é a base de um sistema **RAG (Retrieval-Augmented Generation)**.

### 📚 O que você vai aprender
- Como usar embeddings para busca semântica (retrieval)
- O conceito de RAG e sua arquitetura
- Integração de encoder (NorBERTo) com LLM (geração)
- Avaliação de sistemas de busca e sumarização

---


## 📦 Passo 1 — Instalação

In [ ]:
!pip install transformers datasets torch scikit-learn faiss-cpu openai anthropic -q
print("✅ Pronto!")


## 📚 Passo 2 — Construindo a Base de Conhecimento
Vamos usar o FAQ_BACEN como base de conhecimento. Cada pergunta+resposta será indexada pelo NorBERTo.


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

# Carrega base de conhecimento
print("⏳ Carregando FAQ_BACEN...")
faq = load_dataset("Itau-Unibanco/FAQ_BACEN", split="train")
print(f"✅ {len(faq)} documentos carregados")

# Prepara textos
col_pergunta = faq.column_names[0]
col_resposta = faq.column_names[1] if len(faq.column_names) > 1 else faq.column_names[0]

documentos = []
for ex in faq:
    pergunta = ex[col_pergunta]
    resposta = ex.get(col_resposta, "")
    texto_completo = f"Pergunta: {pergunta}\nResposta: {resposta}" if resposta != pergunta else pergunta
    documentos.append({"pergunta": pergunta, "resposta": resposta, "texto": texto_completo})

print(f"\n📄 Exemplo de documento:")
print(documentos[0]["texto"][:200])


## 🧮 Passo 3 — Indexação Semântica com NorBERTo
Geramos embeddings para todos os documentos e construímos um índice de busca eficiente com FAISS.


In [ ]:
MODEL_NAME = "Itau-Unibanco/NorBERTo-base"
print(f"⏳ Carregando NorBERTo...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME)
encoder.eval()
print(f"✅ NorBERTo pronto!")

def encode_textos(textos, batch_size=32):
    """Gera embeddings para uma lista de textos em lotes."""
    todos_embeddings = []
    for i in range(0, len(textos), batch_size):
        lote = textos[i:i+batch_size]
        inputs = tokenizer(lote, return_tensors="pt", truncation=True,
                          max_length=256, padding=True)
        with torch.no_grad():
            outputs = encoder(**inputs)
        embs = outputs.last_hidden_state[:, 0, :].numpy()
        # Normaliza para similaridade cosseno
        norms = np.linalg.norm(embs, axis=1, keepdims=True)
        embs = embs / (norms + 1e-8)
        todos_embeddings.append(embs)
        if (i // batch_size) % 5 == 0:
            print(f"   Processados: {min(i+batch_size, len(textos))}/{len(textos)}")
    return np.vstack(todos_embeddings)

print("\n⏳ Gerando embeddings para a base de conhecimento...")
textos_base = [d["texto"] for d in documentos]
embeddings_base = encode_textos(textos_base)
print(f"✅ Índice criado: {embeddings_base.shape}")


## 🔎 Passo 4 — Construindo o Retriever com FAISS
O FAISS (Facebook AI Similarity Search) permite busca vetorial eficiente em milhares de documentos:


In [ ]:
import faiss

# Cria índice FAISS (produto interno = similaridade cosseno para vetores normalizados)
DIM = embeddings_base.shape[1]  # 768
index = faiss.IndexFlatIP(DIM)  # Inner Product (= cosseno para vetores normalizados)
index.add(embeddings_base.astype(np.float32))

print(f"✅ Índice FAISS criado!")
print(f"   Documentos indexados: {index.ntotal}")
print(f"   Dimensão dos vetores: {DIM}")

def recuperar_documentos(consulta, top_k=3):
    """Busca os documentos mais relevantes para a consulta."""
    emb_consulta = encode_textos([consulta])
    scores, indices = index.search(emb_consulta.astype(np.float32), top_k)
    resultados = []
    for score, idx in zip(scores[0], indices[0]):
        resultados.append({
            "documento": documentos[idx],
            "score": float(score),
            "indice": int(idx)
        })
    return resultados

# Teste do retriever
print("\n" + "="*60)
print("TESTE DO RETRIEVER SEMÂNTICO")
print("="*60)
consultas_teste = [
    "Como faço para reclamar de uma cobrança indevida?",
    "O que é Pix e como funciona?",
    "Quais são os direitos do consumidor bancário?",
]
for consulta in consultas_teste:
    print(f"\n🔍 Consulta: '{consulta}'")
    resultados = recuperar_documentos(consulta, top_k=2)
    for i, r in enumerate(resultados):
        print(f"  {i+1}. [sim={r['score']:.3f}] {r['documento']['pergunta'][:70]}...")


## 🤖 Passo 5 — Integração com LLM para Geração de Respostas (RAG)
O RAG combina o retriever (NorBERTo) com um gerador (LLM) para responder perguntas com base na base de conhecimento.

> ⚠️ **Nota:** Para usar a API da OpenAI/Anthropic, você precisa de uma chave de API.  
> Se não tiver, pule para o Passo 6 que usa geração local gratuita.


In [ ]:
# ── Opção A: Com API OpenAI (requer chave) ──────────────────────
import os

OPENAI_KEY = ""  # ← cole sua chave aqui, ou deixe vazio para pular

def gerar_resposta_rag_openai(consulta, contexto, api_key):
    """Gera resposta usando RAG com GPT via API OpenAI."""
    try:
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        prompt = f"""Você é um assistente especializado em regulamentação financeira brasileira.
Use APENAS as informações do contexto abaixo para responder à pergunta.
Se a informação não estiver no contexto, diga que não sabe.

CONTEXTO:
{contexto}

PERGUNTA: {consulta}

RESPOSTA:"""
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300, temperature=0.2
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"[Erro na API: {e}]"

# ── Opção B: Geração local simples (sem API) ──────────────────────
def gerar_resposta_local(consulta, docs_recuperados):
    """Resposta simples concatenando os documentos recuperados."""
    resp = f"Com base na base de conhecimento do BACEN:\n\n"
    for i, r in enumerate(docs_recuperados, 1):
        doc = r["documento"]
        resp += f"{i}. {doc['pergunta']}\n   → {doc.get('resposta', '')[:150]}...\n\n"
    return resp

# Pipeline RAG completo
print("="*60)
print("SISTEMA RAG — NorBERTo (retriever) + LLM (gerador)")
print("="*60)

for consulta in consultas_teste:
    print(f"\n❓ Pergunta: {consulta}")
    
    # 1. Retrieval
    docs = recuperar_documentos(consulta, top_k=3)
    contexto = "\n\n".join([
        f"Q: {d['documento']['pergunta']}\nA: {d['documento'].get('resposta','')}"
        for d in docs
    ])
    
    # 2. Geração
    if OPENAI_KEY:
        resposta = gerar_resposta_rag_openai(consulta, contexto, OPENAI_KEY)
    else:
        resposta = gerar_resposta_local(consulta, docs)
    
    print(f"\n💬 Resposta:\n{resposta}")
    print("-"*60)


## 📊 Passo 6 — Avaliação do Sistema de Busca
Vamos avaliar a qualidade do retriever usando métricas de recuperação de informação:


In [ ]:
import matplotlib.pyplot as plt
import random

def calcular_mrr(consultas_com_relevantes, retriever_fn, top_k=10):
    """
    Mean Reciprocal Rank (MRR): avalia se o documento relevante
    aparece entre os primeiros resultados.
    """
    rr_values = []
    for consulta, indice_relevante in consultas_com_relevantes:
        resultados = retriever_fn(consulta, top_k)
        indices = [r['indice'] for r in resultados]
        if indice_relevante in indices:
            rank = indices.index(indice_relevante) + 1
            rr_values.append(1.0 / rank)
        else:
            rr_values.append(0.0)
    return np.mean(rr_values)

# Cria pares (consulta, documento relevante) usando as próprias perguntas do FAQ
# (a pergunta original deve recuperar seu próprio documento)
amostra_avaliacao = [(documentos[i]["pergunta"], i) for i in random.sample(range(len(documentos)), min(50, len(documentos)))]

mrr_semantico = calcular_mrr(amostra_avaliacao, recuperar_documentos)

# Baseline: busca por palavra-chave (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

tfidf_idx = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf_idx.fit_transform([d["texto"] for d in documentos])

def recuperar_tfidf(consulta, top_k=10):
    vec = tfidf_idx.transform([consulta])
    sims = cos_sim(vec, tfidf_matrix)[0]
    indices_top = sims.argsort()[::-1][:top_k]
    return [{"indice": int(i), "score": float(sims[i])} for i in indices_top]

mrr_tfidf = calcular_mrr(amostra_avaliacao, recuperar_tfidf)

# Visualização
fig, ax = plt.subplots(figsize=(8, 5))
sistemas = ["TF-IDF\n(baseline)", "NorBERTo\n(semântico)"]
mrrs = [mrr_tfidf, mrr_semantico]
cores = ["#94A3B8", "#0D9488"]
bars = ax.bar(sistemas, mrrs, color=cores, width=0.4)
for bar, v in zip(bars, mrrs):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.01,
            f"MRR = {v:.3f}", ha='center', fontweight='bold', fontsize=13)
ax.set_title("Qualidade do Retriever: TF-IDF vs NorBERTo (MRR@10)", fontsize=13, fontweight='bold')
ax.set_ylabel("Mean Reciprocal Rank (MRR)")
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("rag_fase5.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 MRR@10 — TF-IDF:   {mrr_tfidf:.4f}")
print(f"   MRR@10 — NorBERTo: {mrr_semantico:.4f}")
print(f"   Ganho:             {mrr_semantico - mrr_tfidf:+.4f}")


## 📝 Avaliação — Fase 5

**Q1.** O que significa RAG em NLP?  
( ) Rapid Answer Generation  
( ) Retrieval-Augmented Generation  
( ) Recursive Attention Grouping  
( ) Relevance-Adjusted Grammar

**Q2.** No sistema RAG desta fase, qual é o papel do NorBERTo?  
( ) Gerador — produz a resposta final em linguagem natural  
( ) Retriever — encontra os documentos mais relevantes por similaridade semântica  
( ) Tokenizador — converte o texto em tokens antes do LLM  
( ) Avaliador — mede a qualidade das respostas geradas

**Q3.** O que é o FAISS (Facebook AI Similarity Search)?  
( ) Um modelo de linguagem para português  
( ) Uma biblioteca para busca eficiente em espaços vetoriais de alta dimensão  
( ) Uma métrica de avaliação de sistemas NLP  
( ) Um dataset de perguntas e respostas em francês

**Q4.** Por que os embeddings são normalizados antes de serem inseridos no FAISS IndexFlatIP?  
( ) Para reduzir o tamanho do índice em disco  
( ) Para que o produto interno (IP) seja equivalente à similaridade cosseno  
( ) Para que o FAISS processe mais rápido  
( ) Porque o FAISS não aceita vetores com valores negativos

**Q5.** O que é o MRR (Mean Reciprocal Rank)?  
( ) A média do F1-Score em todos os documentos  
( ) A média do recíproco do rank do primeiro documento relevante recuperado  
( ) O número médio de documentos recuperados por consulta  
( ) A precisão média dos top-10 resultados

**Q6.** Qual é a principal vantagem do RAG sobre um LLM puro (sem retrieval)?  
( ) O RAG é mais rápido que o LLM  
( ) O RAG ancora as respostas em documentos reais, reduzindo alucinações  
( ) O RAG não precisa de GPU para rodar  
( ) O RAG usa menos memória que o LLM

**Q7.** No prompt do RAG desta fase, por que instruímos o LLM a "usar APENAS as informações do contexto"?  
( ) Para economizar tokens na chamada da API  
( ) Para garantir que as respostas sejam fundamentadas nos documentos recuperados  
( ) Porque o LLM não tem conhecimento sobre finanças  
( ) Para evitar que o modelo traduza as respostas para inglês

**Q8.** O que acontece com a qualidade do sistema RAG se o retriever for ruim?  
( ) O LLM compensa com seu conhecimento geral  
( ) A qualidade das respostas cai, pois o LLM recebe contexto irrelevante  
( ) Não há impacto — o LLM sempre gera boas respostas  
( ) O sistema para de funcionar completamente

**Q9.** IndexFlatIP no FAISS realiza qual tipo de busca?  
( ) Busca aproximada com compressão de vetores  
( ) Busca exata por produto interno (força bruta)  
( ) Busca por distância euclidiana entre clusters  
( ) Busca hierárquica com grafos de vizinhança

**Q10.** Qual das arquiteturas abaixo é um exemplo de modelo ENCODER usado em retrieval semântico?  
( ) GPT-4 (decoder-only)  
( ) T5 (encoder-decoder)  
( ) NorBERTo/BERT (encoder-only)  
( ) Llama 3 (decoder-only)
